# Last-Mile Delivery Route & Fleet Optimization

Explore the synthetic multi-vehicle routing project, compare optimized routes with a nearest-neighbor baseline, and inspect the 25 demand/fleet scenarios.

> All stops and results are synthetic. Install `requirements.txt` from the repository root to use OR-Tools as the primary solver.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists() and (ROOT / 'last-mile-delivery-optimization' / 'src').exists():
    ROOT = ROOT / 'last-mile-delivery-optimization'
sys.path.insert(0, str(ROOT / 'src'))
from generate_data import generate
from optimize_routes import run_all, ORTOOLS_AVAILABLE

print('Primary OR-Tools solver available:', ORTOOLS_AVAILABLE)

In [ ]:
# Regenerate synthetic input data and all scenario outputs.
generate()
results = run_all()
summary = results['summary']
summary.head()

## Scenario-level KPIs

In [ ]:
display(summary[['scenario_id', 'demand_stops', 'fleet_size', 'baseline_distance_km',
                 'optimized_distance_km', 'distance_reduction_pct',
                 'average_route_duration_minutes', 'active_fleet_utilization_pct',
                 'average_shift_utilization_pct', 'solver']].round(2))

print(f"Mean distance reduction vs nearest-neighbor: {summary['distance_reduction_pct'].mean():.2f}%")
print(f"Minimum scenario distance reduction: {summary['distance_reduction_pct'].min():.2f}%")

## Baseline versus optimized distance

Compare the total kilometers from the simple baseline against the selected routing solver for each scenario. Results can differ when the OR-Tools solver is installed versus the fallback.

In [ ]:
plot_data = summary.sort_values(['demand_stops', 'fleet_size']).copy()
plt.figure(figsize=(12, 5))
plt.plot(plot_data['scenario_id'], plot_data['baseline_distance_km'], marker='o', label='Nearest-neighbor baseline')
plt.plot(plot_data['scenario_id'], plot_data['optimized_distance_km'], marker='o', label='Optimized route')
plt.xticks(rotation=75, ha='right')
plt.ylabel('Total distance (km)')
plt.title('Route distance by demand/fleet scenario')
plt.legend()
plt.tight_layout()
plt.show()

## Inspect the 220-stop, 7-vehicle scenario

In [ ]:
showcase = summary.loc[summary['scenario_id'] == 'D220_V7']
display(showcase.T.rename(columns={showcase.index[0]: 'D220_V7'}))

routes = results['routes']
display(routes.loc[routes['scenario_id'] == 'D220_V7'].sort_values('vehicle_id'))

## Route-level utilization

`active_fleet_utilization_pct` measures the share of scheduled vehicles with a non-empty route. It is not the same as shift utilization. `average_shift_utilization_pct` uses estimated route duration divided by shift time.

In [ ]:
vehicle_rows = results['vehicle_utilization']
showcase_routes = vehicle_rows.loc[vehicle_rows['scenario_id'] == 'D220_V7'].sort_values('vehicle_id')
display(showcase_routes)

plt.figure(figsize=(8, 4))
plt.bar(showcase_routes['vehicle_id'], showcase_routes['route_duration_minutes'])
plt.axhline(600, linestyle='--', label='10-hour shift limit')
plt.ylabel('Route duration (minutes)')
plt.title('Estimated route duration by vehicle — D220_V7')
plt.legend()
plt.tight_layout()
plt.show()

## Power BI handoff

Use the CSVs under `data/output/`: `scenario_summary.csv`, `optimized_routes.csv`, `optimized_stop_assignments.csv`, `vehicle_utilization.csv`, and `nearest_neighbor_baseline_routes.csv`. Dashboard suggestions and metric definitions are in `docs/powerbi_dashboard_guide.md`.